In [ ]:
from flask import Flask, request, jsonify, render_template_string
import tensorflow as tf
import numpy as np
import os
import requests
import numpy as np
from flask import Flask, request, render_template_string, jsonify
import numpy as np
import pickle

In [2]:
app = Flask(__name__)
with open(r"C:\onedrive\Desktop\Senior Proejct\Project\tfidf_vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)

In [3]:

# Load the model
MODEL_PATH = r"phishing_model.keras"
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model not found at {MODEL_PATH}")
model = tf.keras.models.load_model(MODEL_PATH)




In [ ]:
# Simple HTML page
HOME_HTML = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Phishing Detection System </title>

    <style>
        @import url('https://fonts.googleapis.com/css2?family=Orbitron:wght@400;600;700&family=Roboto+Mono:wght@300;400;700&display=swap');

        body {
            margin: 0;
            background: #0a0a0f;
            font-family: "Roboto Mono", monospace;
            color: #f6f6f6;
            overflow-x: hidden;
        }

        /* Neon noise overlay */
        .noise {
            pointer-events: none;
            position: fixed;
            width: 100%;
            height: 100%;
            top: 0; left: 0;
            background-image: url('https://i.imgur.com/o9bV8Zf.png');
            opacity: 0.05;
            z-index: 9999;
        }

        /* Scanline overlay */
        .scanlines {
            pointer-events: none;
            position: fixed;
            width: 100%;
            height: 100%;
            top: 0; left: 0;
            background: repeating-linear-gradient(
                to bottom,
                rgba(255,255,255,0.03),
                rgba(255,255,255,0.03) 2px,
                transparent 2px,
                transparent 4px
            );
            z-index: 9998;
        }

        /* Cyberpunk Navbar */
        nav {
            background: #fcee09;
            color: #000;
            padding: 18px 40px;
            font-family: Orbitron, sans-serif;
            font-weight: 700;
            border-bottom: 4px solid cyan;
            display: flex;
            justify-content: space-between;
            letter-spacing: 2px;
            clip-path: polygon(0 0, 96% 0, 100% 100%, 0 100%);
        }

        nav ul {
            list-style: none;
            display: flex;
            gap: 35px;
            margin: 0;
        }
        nav li {
            cursor: pointer;
            transition: 0.2s;
        }
        nav li:hover {
            color: #ff006e;
            text-shadow: 0 0 6px #ff006e, 0 0 12px #ff006e;
        }

        /* Glitch Title */
        h1.glitch {
            font-family: Orbitron, sans-serif;
            font-size: 52px;
            color: #fcee09;
            text-align: center;
            margin-top: 100px;
            font-weight: 700;
            position: relative;
        }

        h1.glitch::before,
        h1.glitch::after {
            content: "PHISHING DETECTION SYSTEM";
            position: absolute;
            left: 0;
            top: 0;
            width: 100%;
        }

        h1.glitch::before {
            color: #00eaff;
            transform: translate(3px, 0);
            opacity: 0.6;
        }

        h1.glitch::after {
            color: #ff006e;
            transform: translate(-3px, 0);
            opacity: 0.6;
        }

        /* Cyan subtitle */
        .subtitle {
            text-align: center;
            color: #00eaff;
            font-size: 18px;
            margin-top: 10px;
            letter-spacing: 1px;
        }

        /* Cyberpunk card box */
        .box {
            background: rgba(15, 15, 25, 0.9);
            width: 650px;
            margin: 60px auto;
            padding: 40px;
            border: 2px solid #fcee09;
            box-shadow: 0 0 15px #fcee09;
            border-radius: 0;
            clip-path: polygon(0 0, 100% 0, 100% 92%, 95% 100%, 0 100%);
            position: relative;
        }

        .box::before {
            content: "";
            position: absolute;
            right: -4px;
            top: 10px;
            height: 40%;
            width: 4px;
            background: #ff006e;
            box-shadow: 0 0 10px #ff006e;
        }

        .box h2 {
            color: #fcee09;
            font-family: Orbitron;
        }

        textarea {
            width: 100%;
            height: 180px;
            background: #050505;
            color: #fcee09;
            border: 2px solid #00eaff;
            border-radius: 0;
            padding: 10px;
            font-size: 14px;
            transition: 0.2s;
            outline: none;
        }

        textarea:focus {
            box-shadow: 0 0 12px #00eaff;
        }

        button {
            margin-top: 15px;
            width: 100%;
            padding: 14px;
            background: #ff006e;
            border: none;
            color: white;
            font-weight: bold;
            font-size: 16px;
            cursor: pointer;
            font-family: Orbitron;
            letter-spacing: 2px;
            transition: 0.3s;
        }

        button:hover {
            background: #fcee09;
            color: #000;
            box-shadow: 0 0 12px #fcee09;
        }

        .result {
            margin-top: 25px;
            background: #111;
            padding: 15px;
            border-left: 4px solid #fcee09;
            color: #fcee09;
            box-shadow: 0 0 12px #fcee09 inset;
            font-size: 15px;
        }
    </style>
</head>

<body>

<div class="noise"></div>
<div class="scanlines"></div>

<nav>
    <div>PHISHING DETECTION SYSTEM by Coated Goders</div>
    <ul>
        <li>HOME</li>
        <li>SCANNER</li>
        <li>API</li>
        <li>ABOUT</li>
    </ul>
</nav>

<h1 class="glitch">PHISHING DETECTION SYSTEM</h1>
<p class="subtitle">Machine Learning Based Email Scanner for Phising Prevention</p>

<div class="box">
    <h2>Email Vector Analyzer</h2>
    <p>Input transmission:</p>

    <form action="/predict" method="post">
        <textarea name="email_text" placeholder="Paste raw email content..."></textarea>
        <button type="submit">EXECUTE SCAN</button>
    </form>

    {% if prediction_label %}
    <div class="result">
        ▸ Prediction: {{ prediction_label }} <br>
        ▸ Probability Index: {{ prediction_prob | round(3) }}
    </div>
    {% endif %}
</div>

</body>
</html>
"""


@app.route('/', methods=['GET', 'POST'])
def home():
    prediction_label = None
    prediction_prob = None
    if request.method == 'POST':
        # get email text from form
        email_text = request.form.get('email_text', '')

        # convert email text to model input
        email_text = request.form.get('email_text', '')
        features = vectorizer.transform([email_text]).toarray().astype(np.float32)
        prediction_prob = float(model.predict(features)[0][0])

        # Convert probability to label
        prediction_label = "Phishing" if prediction_prob >= 0.5 else "Safe"

    return render_template_string(HOME_HTML, prediction_label=prediction_label, prediction_prob=prediction_prob)


@app.route('/predict', methods=['POST'])
def predict():
    try:
        # Check if the request is JSON or form
        if request.is_json:
            data = request.get_json()
            email_text = data['email_text']
        else:
            email_text = request.form.get('email_text', '')
        
        features = vectorizer.transform([email_text]).toarray().astype(np.float32)
        preds = model.predict(features)
        pred_prob = float(model.predict(features)[0][0])
        pred_label = "Phishing" if pred_prob >= 0.5 else "Safe"

        return jsonify({
            "prediction_label": pred_label,
            })
    except Exception as e:
        return jsonify({'error': str(e)}), 500

if __name__ == '__main__':
    app.run(debug=True, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [05/Nov/2025 16:42:10] "GET / HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 430ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step


127.0.0.1 - - [05/Nov/2025 16:42:29] "POST /predict HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step


...

127.0.0.1 - - [05/Nov/2025 16:43:08] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [05/Nov/2025 16:44:48] "GET / HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 170ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step


127.0.0.1 - - [05/Nov/2025 16:45:04] "POST /predict HTTP/1.1" 200 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 139ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step


127.0.0.1 - - [05/Nov/2025 16:45:17] "POST /predict HTTP/1.1" 200 -
